# 5.1 Вероятность, условная вероятность, Байес: код страницы

Аналитика 101 · блок «Дата аналитика» · тема 05 «Вероятность» · страница 5.1.

В ноутбуке три блока кода со страницы: симуляция ста тысяч пользователей, таблица сопряжённости с теоремой Байеса и проверка независимости. Данные учебные: 100 000 визитов из общего набора темы `data/probability/`.

**Как работать.** Выполняйте ячейки по порядку сверху вниз. Блоки самостоятельны: каждый читает файл заново.

Вывод получен на Python 3.13, NumPy 2.5, pandas 3.0 и SciPy 1.18.

## Где лежат данные

Код страницы читает файл по пути от корня папки с материалами: `data/probability/visits.csv`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd
import scipy

root = pathlib.Path.cwd()
while not (root / "data" / "probability").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/probability/
assert (root / "data" / "probability" / "visits.csv").exists(), \
    "Не найдена папка data/probability/. Скачайте общий набор темы 05 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)

Python 3.13.18 | numpy 2.5.3 | pandas 3.0.5 | scipy 1.18.1


## Симуляция: вероятность как доля в большом числе испытаний

### Сто тысяч пользователей: условная доля — маска внутри круга

In [2]:
import numpy as np

rng = np.random.default_rng(51)
n = 100_000
# учебный мир: доли источников и конверсия каждого — как в генераторе visits.csv
source = rng.choice(["органика", "реклама", "рассылка"], n, p=[0.55, 0.30, 0.15])
p_buy = np.select([source == "органика", source == "реклама", source == "рассылка"],
                  [0.025, 0.040, 0.060])
bought = rng.random(n) < p_buy          # у каждого своя монетка с весом p_buy

ad = source == "реклама"
print("P(купил)           ", bought.mean().round(4))
print("P(купил | реклама) ", bought[ad].mean().round(4))   # доля внутри круга «реклама»
print("P(реклама | купил) ", ad[bought].mean().round(4))   # доля внутри круга «купил»

# те же числа по формулам: полная вероятность и Байес
p_b = 0.55 * 0.025 + 0.30 * 0.040 + 0.15 * 0.060
print("по формулам:        ", round(p_b, 4), 0.04, round(0.04 * 0.30 / p_b, 4))

# антифрод: фрод 1%, тест ловит 99% фрода и срабатывает на 1% честных
fraud = rng.random(n) < 0.01
alarm = np.where(fraud, rng.random(n) < 0.99, rng.random(n) < 0.01)
print("сработок", alarm.sum(), "из них фрод", (fraud & alarm).sum())
print("P(фрод | сработка) ", fraud[alarm].mean().round(3))

P(купил)            0.0342
P(купил | реклама)  0.0394
P(реклама | купил)  0.3448
по формулам:         0.0348 0.04 0.3453
сработок 1938 из них фрод 948
P(фрод | сработка)  0.489


Зерно 51, а не 505, как у visits.csv, поэтому числа чуть отличаются от учебного файла: это другая случайная выборка из того же мира. От формул симуляция отходит на десятые доли процентного пункта, у антифрода сильнее: мошеннических транзакций среди 100 000 всего около тысячи.

## Таблица сопряжённости и теорема Байеса

### pd.crosstab: normalize выбирает знаменатель

In [3]:
import pandas as pd

v = pd.read_csv("data/probability/visits.csv")   # 100 000 учебных визитов: source, bought
t = pd.crosstab(v["source"], v["bought"], margins=True, margins_name="всего")
print(t)

# доли по строкам: P(купил | источник) — конверсия каждого источника
print(pd.crosstab(v["source"], v["bought"], normalize="index").round(4))
# доли по столбцам: P(источник | купил) — откуда пришли покупатели
print(pd.crosstab(v["source"], v["bought"], normalize="columns").round(4))

# Байес из той же таблицы: P(реклама | купил) = P(купил | реклама) · P(реклама) / P(купил)
n = t.loc["всего", "всего"]
p_ad = t.loc["реклама", "всего"] / n
p_buy = t.loc["всего", 1] / n
p_buy_ad = t.loc["реклама", 1] / t.loc["реклама", "всего"]
print("Байес:", round(p_buy_ad * p_ad / p_buy, 4),
      "  прямо из столбца:", round(t.loc["реклама", 1] / t.loc["всего", 1], 4))

# полная вероятность: P(купил) = сумма P(купил | источник) · P(источник)
w = v["source"].value_counts(normalize=True)
conv = v.groupby("source")["bought"].mean()
print("полная вероятность:", round((conv * w).sum(), 5), " среднее bought:", v["bought"].mean())

bought        0     1   всего
source                       
органика  53802  1307   55109
рассылка  14146   897   15043
реклама   28676  1172   29848
всего     96624  3376  100000
bought         0       1
source                  
органика  0.9763  0.0237
рассылка  0.9404  0.0596
реклама   0.9607  0.0393


bought         0       1
source                  
органика  0.5568  0.3871
рассылка  0.1464  0.2657
реклама   0.2968  0.3472
Байес: 0.3472   прямо из столбца: 0.3472
полная вероятность: 0.03376  среднее bought: 0.03376


Столбец 1 в долях по строкам — конверсия источника: у рекламы 0,0393. Столбец 1 в долях по столбцам — откуда пришли покупатели: реклама 0,3472. Это два числа из задачи, и Байес переводит первое во второе. Полная вероятность собирает 0,03376 из трёх конверсий.

## Проверка независимости

### Независимость по таблице: доли, ожидаемые числа, хи-квадрат

In [4]:
import pandas as pd
from scipy.stats import chi2_contingency

v = pd.read_csv("data/probability/visits.csv")
t = pd.crosstab(v["source"], v["bought"])

# 1. P(купил | источник) против P(купил): при независимости все строки одинаковы
print(v.groupby("source")["bought"].mean().round(4).to_dict(), round(v["bought"].mean(), 4))

# 2. Сколько было бы в клетках при независимости: n · P(источник) · P(купил)
res = chi2_contingency(t)
print(pd.DataFrame(res.expected_freq, index=t.index, columns=t.columns).round(1))

# 3. Хи-квадрат: насколько наблюдаемая таблица далека от ожидаемой
print(f"хи-квадрат = {res.statistic:.1f}, степеней свободы {res.dof}, p = {res.pvalue:.1e}")

# для сравнения — признак, который с покупкой не связан: чётность номера визита
even = v["visit_id"] % 2 == 0
r2 = chi2_contingency(pd.crosstab(even, v["bought"]))
print(f"чётность: хи-квадрат = {r2.statistic:.2f}, p = {r2.pvalue:.2f}")

{'органика': 0.0237, 'рассылка': 0.0596, 'реклама': 0.0393} 0.0338
bought          0       1
source                   
органика  53248.5  1860.5
рассылка  14535.1   507.9
реклама   28840.3  1007.7
хи-квадрат = 506.8, степеней свободы 2, p = 9.1e-111
чётность: хи-квадрат = 0.33, p = 0.56


При независимости в клетке «реклама, купил» ожидалось бы 1 007,7 визита, а там 1 172; у рассылки разрыв больше: 507,9 против 897. Хи-квадрат 506,8 при двух степенях свободы и p = 9,1·10⁻¹¹¹: при независимости такое расхождение практически не встречается. У чётности номера p = 0,56 — разница в пределах шума. Выбор критерия и чтение p — страницы 6.3 и 6.4.

## Что дальше

Рядом лежит `scene_points.py`: он показывает, как из 100 000 визитов собрана пропорциональная тысяча точек для анимации и интерактива, и насколько сильнее «врёт» обычная случайная тысяча.